# 07 · Génération et attention sur de petits exemples

Ting Tan · EEEA · Oct 2026

Calculer une probabilité du prochain token, générer une courte séquence et lire une matrice d’attention causale.

0–10 min : prédire ; 10–25 : exemple guidé ; 25–45 : exécuter ; 45–65 : une modification ; 65–80 : interpréter ; 80–90 : expliquer et rendre.

## A1 · Prédire avant d’exécuter

Le mot un est suivi de livre 3 fois et de texte 4 fois dans le corpus français. Quelles sont les deux probabilités empiriques du prochain token ?

## A2 · Comprendre l’idée

Le corpus local contient sept courtes phrases fictives par langue. Un token est ici un mot séparé par des espaces. Le modèle bigramme estime le prochain mot à partir du mot courant : P(suivant|courant)=compte(courant,suivant)/somme_suivants compte(courant,suivant). La génération tire un token, l’ajoute et recommence jusqu’à END ou au budget de 12 nouveaux tokens. Le modèle représente les comptes de paires de mots. Les phrases permettent de discuter répétition, cohérence et appui factuel.

La température transforme les probabilités positives : p_T(i)=p(i)^(1/T)/somme_j p(j)^(1/T). Un T positif plus petit favorise le token le plus probable. L’attention constitue un calcul numérique distinct : scores=QKᵀ/sqrt(d), masquage du futur, softmax par ligne, puis multiplication par V. Chaque ligne somme à 1 et les poids futurs valent 0. Ce calcul illustre un composant du Transformer ; le générateur de mots utilise les probabilités bigrammes.

Exécuter une fois la cellule de préparation. Elle localise le dossier du cours et importe les outils fournis.

In [ ]:
from pathlib import Path
import sys
candidates = [q for p in [Path.cwd(), *Path.cwd().parents] for q in [p, p / "ML_Labs_2026"]]
ROOT = next(p for p in candidates if (p / "ml_labs.py").exists())
sys.path.insert(0, str(ROOT))
from ml_labs import run_lab, show, split_data, read_rows
from feedback import check_answer
import numpy as np
LANG = "fr"
print("Course folder:", ROOT.name)
print("Python:", sys.version.split()[0])

## A3 · Exemple guidé

In [ ]:
import json
from models import Bigram, causal_attention
sentences = json.loads((ROOT / "data" / "corpus.json").read_text(encoding="utf-8"))[LANG]
model = Bigram(sentences)
context = "a" if LANG == "en" else "un"
tokens, probabilities = model.distribution(context)
print(list(zip(tokens, probabilities)))
Q = np.array([[1.,0.], [0.,1.], [1.,1.]])
V = np.array([[10.,0.], [0.,10.], [10.,10.]])
attention_output, weights = causal_attention(Q, Q, V)
print(weights)
print("row sums:", weights.sum(axis=1))

Les probabilités valent 3/7 et 4/7. La première ligne d’attention est [1,0,0], donc sa sortie vaut V[0]=[10,0]. La deuxième ligne vaut environ [0,33024,0,66976,0]. Pour T=0,5, les deux probabilités deviennent 9/25 et 16/25.

## B1 · Écrire P(texte|un) à partir du corpus français.

None marque une réponse à compléter. Cette cellule est indépendante de l’expérience du modèle.

In [ ]:
student_probability = None
check_answer(student_probability, 4 / 7, LANG)

<details><summary>Indice 1</summary>

Utiliser le compte de cette paire au numérateur.

</details>

<details><summary>Indice 2</summary>

Diviser par la somme des comptes de toutes les suites du contexte.

</details>

## B2 · Exécuter la référence et lire les axes

Expliquer pourquoi la température change l’échantillonnage en conservant les comptes appris. Lire une ligne d’attention comme moyenne pondérée de V. Comparer les sorties d’une classification et d’une génération.

In [ ]:
baseline = run_lab(7, LANG, ROOT / "outputs" / LANG / "lab07_baseline")
show(baseline)

## B3 · Modifier un élément

Comparer les températures 1 et 0,5 avec le même corpus et les mêmes graines. Examiner d’abord les probabilités, puis les phrases. Distinguer les observations sur la génération et celles sur le calcul séparé d’attention.

In [ ]:
temperature = 0.5
changed = run_lab(7, LANG, ROOT / "outputs" / LANG / "lab07_changed", temperature=temperature)
show(changed)

## C · Exploration facultative : choisir une entrée ou une valeur de paramètre supplémentaire. Prédire son effet, exécuter et justifier l’observation.

## Expliquer le résultat

Remettre la dérivation de 3/7 et 4/7, le calcul de température, l’explication d’une ligne d’attention et l’analyse d’un texte généré.

Ma prédiction :

Mon observation :

Mon explication :

Une question à discuter :